In [ ]:
# SPDX-FileCopyrightText: 2026 Mario Gemoll
# SPDX-License-Identifier: 0BSD

import os
import subprocess

def is_correct_repo() -> bool:
    try:
        result = subprocess.run(
            ["git", "remote", "get-url", "origin"],
            capture_output=True,
            text=True,
            check=True,
        )
        remote_url = result.stdout.strip()
        return remote_url in [
            "https://github.com/mariogemoll/reinforcement-learning.git",
            "git@github.com:mariogemoll/reinforcement-learning.git",
        ]
    except (subprocess.CalledProcessError, FileNotFoundError):
        return False


if not is_correct_repo():
    !git clone https://github.com/mariogemoll/reinforcement-learning.git

if not os.getcwd().endswith("reinforcement-learning/py"):
    %cd reinforcement-learning/py

In [ ]:
%pip install -q gymnax

In [ ]:
import jax
import jax.numpy as jnp
import gymnax
from cartpole_pg import (
    calculate_returns,
    init_mlp_params,
    log_prob_trajectories,
    generate_rollouts,
)
from tqdm.notebook import trange

In [ ]:
num_iterations = 400
num_rollouts = 32
max_steps = 500
lr = 1e-2
seed = 1

In [ ]:
def plot_training_metrics(losses, mean_returns, title="CartPole training metrics"):
    import matplotlib.pyplot as plt

    x = range(len(losses))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(x, losses)
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("Loss")

    axes[1].plot(x, mean_returns)
    axes[1].set_title("Mean return")
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("Return")
    axes[1].set_ylim(0, 550)

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

def run_training(loss_fn):

    env, env_params = gymnax.make("CartPole-v1")
    key = jax.random.PRNGKey(seed)
    key, key_init = jax.random.split(key)
    params = init_mlp_params(key_init, [4, 8, 2])

    losses, mean_returns = [], []

    def train_step(params, key, loss_fn):
        key, subkey = jax.random.split(key)
        rollouts = generate_rollouts(
            subkey, env, env_params, params, num_rollouts, max_steps
        )
        loss, grad = jax.value_and_grad(loss_fn)(params, rollouts)
        params = jax.tree.map(lambda p, g: p - lr * g, params, grad)
        mean_ret = jnp.mean(calculate_returns(rollouts))
        return params, key, loss, mean_ret

    jit_train_step = jax.jit(train_step, static_argnames=("loss_fn",))

    pbar = trange(num_iterations, desc="Training", leave=False)
    for i in pbar:
        params, key, loss, mean_ret = jit_train_step(params, key, loss_fn)
        losses.append(float(loss))
        mean_returns.append(float(mean_ret))
        if (i + 1) % 20 == 0:
            pbar.set_postfix(
                loss=f"{float(loss):.2f}",
                mean_return=f"{float(mean_ret):.1f}",
            )

    plot_training_metrics(losses, mean_returns)
    return params

In [ ]:
%%bash
set -euo pipefail
cd ../ts
if command -v pnpm >/dev/null 2>&1; then
  pnpm i
  pnpm run build:anywidget-cartpole-reinforce
else
  npm install
  npm run build:anywidget-cartpole-reinforce
fi

In [ ]:
def reinforce_loss(params, rollouts):
    log_probs = log_prob_trajectories(params, rollouts)
    returns = calculate_returns(rollouts)
    return -jnp.mean(log_probs * returns)

reinforce_params = run_training(reinforce_loss)

In [ ]:
import base64
from pathlib import Path
from safetensors.numpy import save_file
from IPython.display import display
from cartpole_reinforce import CartPoleReinforceVisualization

def show_policy(params):
    weights_path = Path("cartpole-reinforce-weights.safetensors")
    weights = {}
    for i, layer in enumerate(params):
        weights[f'w{i}'] = layer['w'].T.__array__()
        weights[f'b{i}'] = layer['b'].__array__()
    save_file(weights, weights_path)
    weights_b64 = base64.b64encode(weights_path.read_bytes()).decode("ascii")
    display(CartPoleReinforceVisualization(weights_base64=weights_b64))

In [ ]:
show_policy(reinforce_params)

In [ ]:
def reinforce_with_baseline_loss(params, rollouts):
    log_probs = log_prob_trajectories(params, rollouts)
    returns = calculate_returns(rollouts)
    advantages = (returns - returns.mean()) / (returns.std() + 1e-8)
    return -jnp.mean(log_probs * advantages)

best_params = run_training(reinforce_with_baseline_loss)

In [ ]:
show_policy(best_params)